# MINNNOW GGUF GSM8K Benchmark — Resident Small Models + Adaptive Scaffold v2

This notebook updates the previous MINNOW experiment in three ways:

1. **Higher role-specific token caps** to reduce truncation artifacts.
2. **Optional resident loading of all small GGUF models in GPU memory** instead of lazy-loading one at a time.
3. **Adaptive scaffold v2**, which avoids raw direct answers and routes only between:
   - `direct_finalized`: solver → finalizer
   - `solver_verifier`: solver → independent verifier → finalizer

The goal is to test whether a small-model orchestrated system can remain competitive with a 14B direct baseline while using a similar or lower memory/cost footprint.

## 1. Install dependencies

This tries to install a prebuilt `llama-cpp-python` CUDA wheel. If CUDA wheel installation fails, it falls back to CPU wheel unless you explicitly allow a source build.

In [1]:
#@title Fast install GGUF inference dependencies

USE_PREBUILT_LLAMA_CPP = True  #@param {type:"boolean"}
CUDA_WHEEL = "auto"            #@param ["auto", "cu118", "cu121", "cu122", "cu123", "cu124", "cu125", "cu130", "cu132", "cpu"]
ALLOW_SOURCE_BUILD_IF_WHEEL_FAILS = False  #@param {type:"boolean"}

import os, re, sys, subprocess

LLAMA_CPP_GPU_ACCEL = False

def run(cmd, env=None, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, env=env)

def detect_cuda_wheel():
    supported = ["cu118", "cu121", "cu122", "cu123", "cu124", "cu125", "cu130", "cu132"]
    for cmd in ["nvcc --version", "nvidia-smi"]:
        try:
            out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
            m = re.search(r"release\s+(\d+)\.(\d+)", out) or re.search(r"CUDA Version:\s+(\d+)\.(\d+)", out)
            if m:
                major, minor = int(m.group(1)), int(m.group(2))
                exact = f"cu{major}{minor}"
                if exact in supported:
                    return exact
                if major == 12:
                    candidates = [t for t in supported if t.startswith("cu12") and int(t[-2:]) <= minor]
                    return candidates[-1] if candidates else "cu124"
                if major == 13:
                    candidates = [t for t in supported if t.startswith("cu13") and int(t[-2:]) <= minor]
                    return candidates[-1] if candidates else "cu130"
        except Exception:
            pass
    return "cpu"

run(f"{sys.executable} -m pip install -q -U pip setuptools wheel")
run(f"{sys.executable} -m pip install -q -U datasets huggingface_hub pandas tqdm psutil pynvml")

if USE_PREBUILT_LLAMA_CPP:
    wheel_tag = detect_cuda_wheel() if CUDA_WHEEL == "auto" else CUDA_WHEEL
    print(f"Selected llama-cpp-python wheel target: {wheel_tag}")
    index = f"https://abetlen.github.io/llama-cpp-python/whl/{wheel_tag}"
    cmd = (
        f"{sys.executable} -m pip install -q -U --prefer-binary --only-binary=:all: "
        f"llama-cpp-python --extra-index-url {index}"
    )
    result = run(cmd, check=False)
    if result.returncode == 0:
        LLAMA_CPP_GPU_ACCEL = wheel_tag != "cpu"
    else:
        print("Prebuilt selected wheel failed. Falling back to CPU wheel.")
        cpu_cmd = (
            f"{sys.executable} -m pip install -q -U --prefer-binary --only-binary=:all: "
            f"llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu"
        )
        cpu_result = run(cpu_cmd, check=False)
        if cpu_result.returncode == 0:
            LLAMA_CPP_GPU_ACCEL = False
        elif ALLOW_SOURCE_BUILD_IF_WHEEL_FAILS:
            env = os.environ.copy()
            env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
            env["FORCE_CMAKE"] = "1"
            run(f"{sys.executable} -m pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python", env=env)
            LLAMA_CPP_GPU_ACCEL = True
        else:
            raise RuntimeError("Could not install llama-cpp-python wheel. Set ALLOW_SOURCE_BUILD_IF_WHEEL_FAILS=True as a last resort.")
else:
    env = os.environ.copy()
    env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    env["FORCE_CMAKE"] = "1"
    run(f"{sys.executable} -m pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python", env=env)
    LLAMA_CPP_GPU_ACCEL = True

print("LLAMA_CPP_GPU_ACCEL =", LLAMA_CPP_GPU_ACCEL)

$ /usr/bin/python3 -m pip install -q -U pip setuptools wheel
$ /usr/bin/python3 -m pip install -q -U datasets huggingface_hub pandas tqdm psutil pynvml
Selected llama-cpp-python wheel target: cu124
$ /usr/bin/python3 -m pip install -q -U --prefer-binary --only-binary=:all: llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
LLAMA_CPP_GPU_ACCEL = True


## 2. Configuration

Key changes from the previous notebook:

- Direct/solver max tokens increased from `384` to `768`.
- Verifier max tokens increased from `320` to `512`.
- Finalizer capped at `64`.
- Small models can be loaded resident in GPU memory using `PRELOAD_SMALL_MODELS_IN_GPU=True`.

If resident loading OOMs, the notebook automatically falls back to lazy loading.

In [2]:
#@title Experiment configuration

import os, re, gc, json, time, math, random, fnmatch, subprocess
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download
from llama_cpp import Llama

# -------------------------
# Benchmark settings
# -------------------------
N_SAMPLES = 100         # Start with 20 if debugging, then use 100.
SEED = 42
DATASET_SPLIT = "test"

# -------------------------
# llama.cpp inference settings
# -------------------------
N_CTX = 2048
N_BATCH = 256
N_THREADS = max(2, os.cpu_count() or 2)
N_GPU_LAYERS_SMALL = -1 if globals().get("LLAMA_CPP_GPU_ACCEL", True) else 0
N_GPU_LAYERS_LARGE = -1 if globals().get("LLAMA_CPP_GPU_ACCEL", True) else 0

TEMPERATURE = 0.0
TOP_P = 0.95

# Increased caps to reduce truncation artifacts.
MAX_TOKENS_DIRECT = 768
MAX_TOKENS_SOLVER = 768
MAX_TOKENS_VERIFIER = 512
MAX_TOKENS_FINALIZER = 64
MAX_TOKENS_ROUTER = 160
MAX_TOKENS_LARGE_BASELINE = 768

# Resident loading keeps all small models loaded together.
# This should be feasible if 3 small Q4_K_M models fit in GPU memory.
PRELOAD_SMALL_MODELS_IN_GPU = True  #@param {type:"boolean"}
FALLBACK_TO_LAZY_IF_RESIDENT_OOM = True  #@param {type:"boolean"}

# Optional larger baseline.
RUN_LARGE_BASELINE = True  #@param {type:"boolean"}
RUN_LARGE_BASELINE_FINALIZED = True  #@param {type:"boolean"}

# Cost proxy. Adjust to your target deployment hardware.
GPU_HOURLY_COST_USD = 0.35

CACHE_DIR = Path("/content/hf_gguf_cache")
RESULTS_DIR = Path("/content/minnow_gguf_results_v2")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "qwen4b_instruct": {
        "repo_id": "unsloth/Qwen3-4B-Instruct-2507-GGUF",
        "filename_patterns": ["*Q4_K_M.gguf", "*UD-Q4_K_XL.gguf"],
        "description": "Router / verifier / general instruction model",
        "size_class": "small",
    },
    "phi4_mini_reasoning": {
        "repo_id": "unsloth/Phi-4-mini-reasoning-GGUF",
        "filename_patterns": ["*Q4_K_M.gguf", "*UD-Q4_K_XL.gguf"],
        "description": "Primary math solver",
        "size_class": "small",
    },
    "phi4_mini_instruct": {
        "repo_id": "unsloth/Phi-4-mini-instruct-GGUF",
        "filename_patterns": ["*Q4_K_M.gguf", "*UD-Q4_K_XL.gguf"],
        "description": "Final answer synthesizer",
        "size_class": "small",
    },
    "phi4_reasoning_14b": {
        "repo_id": "bartowski/microsoft_Phi-4-reasoning-GGUF",
        "filename_patterns": ["*Q4_K_M.gguf"],
        "description": "Optional larger 14B reasoning baseline",
        "size_class": "large",
    },
}

ROLE_MODEL = {
    "small_direct": "phi4_mini_reasoning",
    "solver": "phi4_mini_reasoning",
    "verifier": "qwen4b_instruct",
    "router": "qwen4b_instruct",
    "finalizer": "phi4_mini_instruct",
    "large_baseline": "phi4_reasoning_14b",
}

SMALL_RESIDENT_KEYS = sorted(set([
    ROLE_MODEL["small_direct"],
    ROLE_MODEL["solver"],
    ROLE_MODEL["verifier"],
    ROLE_MODEL["router"],
    ROLE_MODEL["finalizer"],
]))

print("Small resident candidates:", SMALL_RESIDENT_KEYS)
print("Configured models:")
for key, spec in MODELS.items():
    print(f"- {key}: {spec['repo_id']} | patterns={spec['filename_patterns']} | {spec['description']}")

Small resident candidates: ['phi4_mini_instruct', 'phi4_mini_reasoning', 'qwen4b_instruct']
Configured models:
- qwen4b_instruct: unsloth/Qwen3-4B-Instruct-2507-GGUF | patterns=['*Q4_K_M.gguf', '*UD-Q4_K_XL.gguf'] | Router / verifier / general instruction model
- phi4_mini_reasoning: unsloth/Phi-4-mini-reasoning-GGUF | patterns=['*Q4_K_M.gguf', '*UD-Q4_K_XL.gguf'] | Primary math solver
- phi4_mini_instruct: unsloth/Phi-4-mini-instruct-GGUF | patterns=['*Q4_K_M.gguf', '*UD-Q4_K_XL.gguf'] | Final answer synthesizer
- phi4_reasoning_14b: bartowski/microsoft_Phi-4-reasoning-GGUF | patterns=['*Q4_K_M.gguf'] | Optional larger 14B reasoning baseline


## 3. GGUF utilities: download, resident loading, lazy fallback, and generation

In [3]:
#@title GGUF model resolver, resident loader, lazy fallback, and generation

try:
    import pynvml
    pynvml.nvmlInit()
    _NVML_AVAILABLE = True
except Exception:
    _NVML_AVAILABLE = False

_resident_llms = {}
_current_llm = None
_current_model_key = None
_call_log = []
_resolved_files = {}
_model_paths = {}

def gpu_mem_used_mb():
    if not _NVML_AVAILABLE:
        return None
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return round(info.used / (1024 ** 2), 1)
    except Exception:
        return None

def resolve_gguf_filename(model_key: str) -> str:
    if model_key in _resolved_files:
        return _resolved_files[model_key]
    spec = MODELS[model_key]
    files = list_repo_files(repo_id=spec["repo_id"], repo_type="model")
    ggufs = [f for f in files if f.lower().endswith(".gguf")]
    if not ggufs:
        raise FileNotFoundError(f"No .gguf files found in {spec['repo_id']}")
    for pat in spec["filename_patterns"]:
        matches = [f for f in ggufs if fnmatch.fnmatch(f, pat)]
        single_file_matches = [m for m in matches if "-0000" not in m]
        if single_file_matches:
            matches = single_file_matches
        if matches:
            chosen = sorted(matches, key=lambda x: (len(x), x))[0]
            _resolved_files[model_key] = chosen
            return chosen
    raise FileNotFoundError(
        f"Could not match patterns {spec['filename_patterns']} in {spec['repo_id']}. "
        f"Available GGUFs include: {ggufs[:30]}"
    )

def download_model(model_key: str) -> str:
    if model_key in _model_paths and Path(_model_paths[model_key]).exists():
        return _model_paths[model_key]
    spec = MODELS[model_key]
    filename = resolve_gguf_filename(model_key)
    print(f"Downloading/resolving {model_key}: {spec['repo_id']} :: {filename}")
    path = hf_hub_download(
        repo_id=spec["repo_id"],
        filename=filename,
        cache_dir=str(CACHE_DIR),
        resume_download=True,
        local_files_only=False,
    )
    _model_paths[model_key] = path
    return path

def n_gpu_layers_for(model_key: str):
    return N_GPU_LAYERS_LARGE if MODELS[model_key].get("size_class") == "large" else N_GPU_LAYERS_SMALL

def make_llama(model_key: str):
    model_path = download_model(model_key)
    before = gpu_mem_used_mb()
    t0 = time.time()
    llm = Llama(
        model_path=model_path,
        n_ctx=N_CTX,
        n_batch=N_BATCH,
        n_threads=N_THREADS,
        n_gpu_layers=n_gpu_layers_for(model_key),
        verbose=False,
    )
    load_s = time.time() - t0
    after = gpu_mem_used_mb()
    print(f"Loaded {model_key} in {load_s:.1f}s | GPU MB before={before}, after={after}")
    return llm

def release_lazy_model():
    global _current_llm, _current_model_key
    if _current_llm is not None:
        print(f"Unloading lazy model {_current_model_key}")
    _current_llm = None
    _current_model_key = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def release_all_models():
    global _resident_llms, _current_llm, _current_model_key
    if _resident_llms:
        print("Releasing resident models:", list(_resident_llms.keys()))
    _resident_llms = {}
    _current_llm = None
    _current_model_key = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass
    time.sleep(1)

def preload_resident_models(model_keys):
    """Try to keep all small worker models resident at once."""
    global PRELOAD_SMALL_MODELS_IN_GPU, _resident_llms
    if not PRELOAD_SMALL_MODELS_IN_GPU:
        print("Resident preload disabled.")
        return
    print("Attempting resident preload:", model_keys)
    try:
        release_all_models()
        for key in model_keys:
            before = gpu_mem_used_mb()
            _resident_llms[key] = make_llama(key)
            after = gpu_mem_used_mb()
            print(f"Resident model {key}: GPU MB before={before}, after={after}")
        print("Resident preload succeeded. Resident keys:", list(_resident_llms.keys()))
    except Exception as e:
        print("Resident preload failed:", repr(e))
        if FALLBACK_TO_LAZY_IF_RESIDENT_OOM:
            print("Falling back to lazy loading.")
            release_all_models()
            PRELOAD_SMALL_MODELS_IN_GPU = False
        else:
            raise

def load_model(model_key: str):
    global _current_llm, _current_model_key
    if model_key in _resident_llms:
        return _resident_llms[model_key]
    if _current_llm is not None and _current_model_key == model_key:
        return _current_llm
    release_lazy_model()
    print(f"Lazy-loading {model_key}")
    _current_llm = make_llama(model_key)
    _current_model_key = model_key
    return _current_llm

def unload_model():
    """Backward-compatible helper: unload only lazy model, keep resident models."""
    release_lazy_model()

def messages_to_fallback_prompt(messages):
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"[{role.upper()}]\n{content}")
    parts.append("[ASSISTANT]\n")
    return "\n\n".join(parts)

def chat_gguf(model_key: str, messages, max_tokens: int, temperature: float = TEMPERATURE, top_p: float = TOP_P, stop=None):
    llm = load_model(model_key)
    start = time.time()
    mem_before = gpu_mem_used_mb()
    try:
        out = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            stop=stop,
        )
        text = out["choices"][0]["message"]["content"]
        usage = out.get("usage", {}) or {}
    except Exception:
        prompt = messages_to_fallback_prompt(messages)
        out = llm(
            prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            stop=stop or ["[USER]", "<|im_end|>", "</s>"],
        )
        text = out["choices"][0]["text"].strip()
        usage = out.get("usage", {}) or {}
    elapsed = time.time() - start
    mem_after = gpu_mem_used_mb()
    approx_prompt_chars = sum(len(m.get("content", "")) for m in messages)
    _call_log.append({
        "model_key": model_key,
        "elapsed_s": elapsed,
        "gpu_mem_before_mb": mem_before,
        "gpu_mem_after_mb": mem_after,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "approx_prompt_chars": approx_prompt_chars,
        "approx_completion_chars": len(text),
        "max_tokens_requested": max_tokens,
        "resident": model_key in _resident_llms,
    })
    return text

print("GPU used MB:", gpu_mem_used_mb())

GPU used MB: 447.3


## 4. Load GSM8K and scoring helpers

This still uses numeric extraction, but the updated experiments add direct-finalized baselines to separate reasoning quality from formatting/extraction artifacts.

In [4]:
#@title Dataset and scoring helpers

random.seed(SEED)

def load_gsm8k_sample(n=N_SAMPLES, seed=SEED):
    ds = load_dataset("openai/gsm8k", "main", split=DATASET_SPLIT)
    ds = ds.shuffle(seed=seed).select(range(n))
    return list(ds)

_NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")

def normalize_number_str(x):
    if x is None:
        return None
    x = str(x).strip().replace(",", "")
    x = re.sub(r"\.$", "", x)
    return x

def extract_gsm8k_gold(answer_text: str):
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", answer_text)
    if m:
        return normalize_number_str(m.group(1))
    nums = _NUM_RE.findall(answer_text)
    return normalize_number_str(nums[-1]) if nums else None

def extract_model_answer(text: str):
    if not text:
        return None
    patterns = [
        r"FINAL_ANSWER\s*[:=]\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"FINAL\s*[:=]\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"Final answer\s*[:=]\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"The answer is\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"Answer\s*[:=]\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return normalize_number_str(m.group(1))
    nums = _NUM_RE.findall(text)
    return normalize_number_str(nums[-1]) if nums else None

def output_has_final_answer(text: str):
    return bool(text and re.search(r"FINAL_ANSWER\s*[:=]\s*", text, flags=re.IGNORECASE))

def numbers_equal(a, b, tol=1e-6):
    a = normalize_number_str(a)
    b = normalize_number_str(b)
    if a is None or b is None:
        return False
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return a == b

examples = load_gsm8k_sample()
print(f"Loaded {len(examples)} GSM8K examples")
print("Sample question:\n", examples[0]["question"][:500])
print("Gold final:", extract_gsm8k_gold(examples[0]["answer"]))

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 100 GSM8K examples
Sample question:
 Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.
Gold final: 109


## 5. Prompt templates and agent functions

Adaptive scaffold v2 avoids raw direct answers. Even easy questions go through a short finalizer, so the comparison against solver-verifier is not dominated by parsing failures.

In [5]:
#@title Prompt templates and agent functions

SOLVER_SYSTEM = """You are a careful math solver for GSM8K-style grade-school math.
Solve compactly. Use no more than 8 short reasoning lines.
Do not repeat the full problem.
End with exactly one line in this format:
FINAL_ANSWER: <number>
Do not include units in the final answer line. Do not write anything after FINAL_ANSWER."""

VERIFIER_SYSTEM = """You are a strict independent math verifier.
First solve the problem independently, without trusting the candidate solution.
Then compare your independent answer with the candidate.
If the candidate is wrong, use your independent answer.
Keep the verification concise.
End with exactly one line in this format:
FINAL_ANSWER: <number>
Do not include units in the final answer line. Do not write anything after FINAL_ANSWER."""

FINALIZER_SYSTEM = """You are a final-answer extractor.
Given a math problem and model outputs, return only the final numeric answer.
Prefer the verifier's final answer if it exists; otherwise use the solver's final answer.
Return exactly one line:
FINAL_ANSWER: <number>"""

ROUTER_SYSTEM_V2 = """You are a Minnow orchestrator for GSM8K-style math problems.
Return JSON only. No markdown.

Choose exactly one workflow:
- "direct_finalized": use this only for one-step arithmetic with very low ambiguity.
- "solver_verifier": use this for multi-step arithmetic, rates, ratios, percentages, comparisons, remaining/left/after problems, unit conversions, or any ambiguous wording.

Never choose raw direct. There is no raw direct workflow.

Schema:
{
  "task_type": "math_reasoning",
  "difficulty": "easy|medium|hard",
  "workflow": "direct_finalized|solver_verifier",
  "confidence": 0.0,
  "reason": "brief reason",
  "max_rounds": 1
}

Rules:
- If confidence < 0.85, choose solver_verifier.
- If the problem has more than 3 numbers, choose solver_verifier.
- If the problem contains percent, ratio, rate, per, each, more than, less than, twice, triple, half, remaining, left, after, before, total, altogether, or difference, choose solver_verifier.
"""

COMPLEXITY_KEYWORDS = [
    "percent", "%", "ratio", "rate", "per ", "each", "more than", "less than",
    "twice", "triple", "three times", "half", "quarter", "remaining", "left",
    "after", "before", "total", "altogether", "difference", "every",
    "combined", "spent", "saved", "earn", "earned", "cost", "price", "change",
    "increase", "decrease", "fewer", "greater", "less", "more", "times",
]

def heuristic_requires_verifier(question: str):
    q = " " + question.lower() + " "
    nums = _NUM_RE.findall(question)
    if len(nums) > 3:
        return True, f"{len(nums)} numbers"
    hits = [kw for kw in COMPLEXITY_KEYWORDS if kw in q]
    if hits:
        return True, "keyword:" + ",".join(hits[:5])
    if len(question.split()) > 55:
        return True, "long_question"
    return False, "simple_heuristic"

def direct_small(question: str):
    messages = [
        {"role": "system", "content": SOLVER_SYSTEM},
        {"role": "user", "content": question},
    ]
    return chat_gguf(ROLE_MODEL["small_direct"], messages, max_tokens=MAX_TOKENS_DIRECT)

def direct_small_finalized(question: str):
    solver_out = direct_small(question)
    final_user = f"Problem:\n{question}\n\nSolver output:\n{solver_out}"
    final_out = chat_gguf(
        ROLE_MODEL["finalizer"],
        [
            {"role": "system", "content": FINALIZER_SYSTEM},
            {"role": "user", "content": final_user},
        ],
        max_tokens=MAX_TOKENS_FINALIZER,
    )
    return {
        "solver_output": solver_out,
        "final_output": final_out,
    }

def direct_large(question: str):
    messages = [
        {"role": "system", "content": SOLVER_SYSTEM},
        {"role": "user", "content": question},
    ]
    return chat_gguf(ROLE_MODEL["large_baseline"], messages, max_tokens=MAX_TOKENS_LARGE_BASELINE)

def direct_large_finalized(question: str):
    large_out = direct_large(question)
    final_user = f"Problem:\n{question}\n\nLarge model output:\n{large_out}"
    final_out = chat_gguf(
        ROLE_MODEL["finalizer"],
        [
            {"role": "system", "content": FINALIZER_SYSTEM},
            {"role": "user", "content": final_user},
        ],
        max_tokens=MAX_TOKENS_FINALIZER,
    )
    return {
        "large_output": large_out,
        "final_output": final_out,
    }

def solver_verifier(question: str):
    solver_out = chat_gguf(
        ROLE_MODEL["solver"],
        [
            {"role": "system", "content": SOLVER_SYSTEM},
            {"role": "user", "content": question},
        ],
        max_tokens=MAX_TOKENS_SOLVER,
    )
    verifier_user = (
        f"Problem:\n{question}\n\n"
        f"Candidate solution:\n{solver_out}\n\n"
        "Solve independently first, compare with the candidate, and return the corrected final answer."
    )
    verifier_out = chat_gguf(
        ROLE_MODEL["verifier"],
        [
            {"role": "system", "content": VERIFIER_SYSTEM},
            {"role": "user", "content": verifier_user},
        ],
        max_tokens=MAX_TOKENS_VERIFIER,
    )
    final_user = f"Problem:\n{question}\n\nSolver output:\n{solver_out}\n\nVerifier output:\n{verifier_out}"
    final_out = chat_gguf(
        ROLE_MODEL["finalizer"],
        [
            {"role": "system", "content": FINALIZER_SYSTEM},
            {"role": "user", "content": final_user},
        ],
        max_tokens=MAX_TOKENS_FINALIZER,
    )
    return {
        "solver_output": solver_out,
        "verifier_output": verifier_out,
        "final_output": final_out,
    }

def safe_json_loads(text: str):
    if not text:
        return None
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    m = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if m:
        cleaned = m.group(0)
    try:
        return json.loads(cleaned)
    except Exception:
        return None

def adaptive_scaffold_v2(question: str):
    router_out = chat_gguf(
        ROLE_MODEL["router"],
        [
            {"role": "system", "content": ROUTER_SYSTEM_V2},
            {"role": "user", "content": f"Problem:\n{question}\n\nReturn the workflow JSON."},
        ],
        max_tokens=MAX_TOKENS_ROUTER,
    )
    scaffold = safe_json_loads(router_out) or {
        "task_type": "math_reasoning",
        "difficulty": "medium",
        "workflow": "solver_verifier",
        "confidence": 0.0,
        "reason": "json_parse_failed",
        "max_rounds": 1,
        "fallback": True,
    }

    workflow = scaffold.get("workflow", "solver_verifier")
    if workflow == "direct":
        workflow = "direct_finalized"

    confidence = scaffold.get("confidence", 0.0)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.0

    heuristic_verifier, heuristic_reason = heuristic_requires_verifier(question)
    override_reason = None

    if workflow == "direct_finalized" and confidence < 0.85:
        workflow = "solver_verifier"
        override_reason = f"low_confidence:{confidence}"

    if workflow == "direct_finalized" and heuristic_verifier:
        workflow = "solver_verifier"
        override_reason = f"heuristic_requires_verifier:{heuristic_reason}"

    if workflow == "direct_finalized":
        out = direct_small_finalized(question)
        final_text = out.get("final_output", "")
        if extract_model_answer(final_text) is None:
            sv = solver_verifier(question)
            sv.update({
                "router_output": router_out,
                "scaffold": scaffold,
                "chosen_workflow": "solver_verifier",
                "override_reason": "direct_finalized_unparseable_escalation",
            })
            return sv
        out.update({
            "router_output": router_out,
            "scaffold": scaffold,
            "chosen_workflow": "direct_finalized",
            "override_reason": override_reason,
        })
        return out

    sv = solver_verifier(question)
    sv.update({
        "router_output": router_out,
        "scaffold": scaffold,
        "chosen_workflow": "solver_verifier",
        "override_reason": override_reason,
    })
    return sv

## 6. Evaluation runner

In [6]:
#@title Evaluation runner

def reset_call_log():
    global _call_log
    _call_log = []

def summarize_calls(calls):
    elapsed = sum(c.get("elapsed_s", 0) or 0 for c in calls)
    total_tokens = sum(c.get("total_tokens") or 0 for c in calls)
    completion_tokens = sum(c.get("completion_tokens") or 0 for c in calls)
    approx_chars = sum((c.get("approx_prompt_chars") or 0) + (c.get("approx_completion_chars") or 0) for c in calls)
    mems = [c.get("gpu_mem_after_mb") for c in calls if c.get("gpu_mem_after_mb") is not None]
    return {
        "n_model_calls": len(calls),
        "model_call_elapsed_s": elapsed,
        "total_tokens_reported": total_tokens if total_tokens else None,
        "completion_tokens_reported": completion_tokens if completion_tokens else None,
        "approx_total_chars": approx_chars,
        "peak_gpu_mem_after_mb": max(mems) if mems else None,
        "models_called": ",".join(c.get("model_key", "") for c in calls),
        "resident_calls": sum(1 for c in calls if c.get("resident")),
        "max_tokens_requested_sum": sum(c.get("max_tokens_requested") or 0 for c in calls),
    }

def normalize_mode_output(mode_name, raw):
    if isinstance(raw, dict):
        final_text = raw.get("final_output") or raw.get("verifier_output") or raw.get("solver_output") or raw.get("large_output") or ""
        detail = raw
    else:
        final_text = raw
        detail = {"final_output": raw}
    return final_text, detail

def run_mode(mode_name: str, mode_fn, examples):
    rows = []
    jsonl_path = RESULTS_DIR / f"{mode_name}_details.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for idx, ex in enumerate(tqdm(examples, desc=mode_name)):
            q = ex["question"]
            gold = extract_gsm8k_gold(ex["answer"])
            reset_call_log()
            t0 = time.time()
            try:
                raw = mode_fn(q)
                error = None
            except Exception as e:
                raw = ""
                error = repr(e)
            wall_s = time.time() - t0
            final_text, detail = normalize_mode_output(mode_name, raw)
            pred = extract_model_answer(final_text)
            correct = numbers_equal(pred, gold)
            call_summary = summarize_calls(_call_log)
            est_cost = (wall_s / 3600.0) * GPU_HOURLY_COST_USD
            row = {
                "mode": mode_name,
                "idx": idx,
                "question": q,
                "gold": gold,
                "pred": pred,
                "correct": correct,
                "has_final_answer": output_has_final_answer(final_text),
                "error": error,
                "wall_s": wall_s,
                "estimated_gpu_cost_usd": est_cost,
                **call_summary,
            }
            if isinstance(detail, dict):
                row["chosen_workflow"] = detail.get("chosen_workflow")
                row["override_reason"] = detail.get("override_reason")
                row["router_workflow"] = (detail.get("scaffold") or {}).get("workflow") if isinstance(detail.get("scaffold"), dict) else None
                row["router_confidence"] = (detail.get("scaffold") or {}).get("confidence") if isinstance(detail.get("scaffold"), dict) else None

            rows.append(row)
            f.write(json.dumps({**row, "detail": detail, "call_log": _call_log}, ensure_ascii=False) + "\n")
            print(f"[{mode_name}] {idx+1}/{len(examples)} pred={pred} gold={gold} correct={correct} wall={wall_s:.1f}s calls={row['n_model_calls']}")
    df = pd.DataFrame(rows)
    csv_path = RESULTS_DIR / f"{mode_name}_summary.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path}")
    print(f"Saved {jsonl_path}")
    return df

def aggregate_results(dfs):
    rows = []
    for name, df in dfs.items():
        if df.empty:
            continue
        n = len(df)
        correct = int(df["correct"].sum())
        total_wall = df["wall_s"].sum()
        total_cost = df["estimated_gpu_cost_usd"].sum()
        rows.append({
            "mode": name,
            "n": n,
            "correct": correct,
            "accuracy": correct / n if n else None,
            "total_wall_s": total_wall,
            "avg_wall_s": df["wall_s"].mean(),
            "avg_model_calls": df["n_model_calls"].mean(),
            "total_estimated_gpu_cost_usd": total_cost,
            "cost_per_correct_usd": total_cost / correct if correct else None,
            "avg_peak_gpu_mem_after_mb": df["peak_gpu_mem_after_mb"].dropna().mean() if "peak_gpu_mem_after_mb" in df else None,
            "final_answer_rate": df["has_final_answer"].mean() if "has_final_answer" in df else None,
            "avg_completion_tokens_reported": df["completion_tokens_reported"].dropna().mean() if "completion_tokens_reported" in df else None,
        })
    return pd.DataFrame(rows).sort_values(["accuracy", "cost_per_correct_usd"], ascending=[False, True])

## 7. Pre-download and optionally resident-load small models

This is the key speed/memory test. If the three small Q4 models fit together, all small-model calls avoid repeated reload overhead.

In [ ]:
#@title Pre-download and resident-load small GGUF models

PRE_DOWNLOAD_SMALL_MODELS = True  #@param {type:"boolean"}

if PRE_DOWNLOAD_SMALL_MODELS:
    for key in SMALL_RESIDENT_KEYS:
        download_model(key)
    print("Resolved files:", json.dumps(_resolved_files, indent=2))

if PRELOAD_SMALL_MODELS_IN_GPU:
    preload_resident_models(SMALL_RESIDENT_KEYS)
else:
    print("Skipping resident preload; notebook will lazy-load models.")
print("GPU used MB after preload:", gpu_mem_used_mb())

## 8. Run small-model benchmarks

Recommended first run:

- `N_SAMPLES = 20` while debugging resident loading.
- `N_SAMPLES = 100` once stable.

The most important comparison is now:

```text
small_direct_finalized
vs
solver_verifier
vs
adaptive_scaffold_v2
```

This isolates how much of the previous improvement came from final answer formatting versus actual verification.

In [ ]:
#@title Run small-model benchmarks

results = {}

results["small_direct"] = run_mode("small_direct", direct_small, examples)
unload_model()

results["small_direct_finalized"] = run_mode("small_direct_finalized", direct_small_finalized, examples)
unload_model()

results["solver_verifier"] = run_mode("solver_verifier", solver_verifier, examples)
unload_model()

results["adaptive_scaffold_v2"] = run_mode("adaptive_scaffold_v2", adaptive_scaffold_v2, examples)
unload_model()

summary = aggregate_results(results)
summary_path = RESULTS_DIR / "aggregate_summary.csv"
summary.to_csv(summary_path, index=False)
summary

## 9. Optional: run 14B baseline

This will usually need more VRAM and may force lazy loading. It can be run after the small models finish.

In [ ]:
#@title Optional: run 14B baseline

if RUN_LARGE_BASELINE:
    results["large_14b_direct"] = run_mode("large_14b_direct", direct_large, examples)
    unload_model()

if RUN_LARGE_BASELINE_FINALIZED:
    results["large_14b_direct_finalized"] = run_mode("large_14b_direct_finalized", direct_large_finalized, examples)
    unload_model()

if RUN_LARGE_BASELINE or RUN_LARGE_BASELINE_FINALIZED:
    summary = aggregate_results(results)
    summary_path = RESULTS_DIR / "aggregate_summary_with_large.csv"
    summary.to_csv(summary_path, index=False)
    display(summary)
else:
    print("Large baseline disabled. Set RUN_LARGE_BASELINE and/or RUN_LARGE_BASELINE_FINALIZED=True in config.")

## 10. Diagnostics

These cells help answer:

- Did higher token caps reduce truncation?
- Did direct-finalization close the gap?
- Did resident loading improve speed?
- Is adaptive scaffold v2 still choosing direct-finalized too often?

In [ ]:
#@title Diagnostics and pairwise comparison

def print_mode_diagnostics(results):
    for name, df in results.items():
        print("\n" + "="*80)
        print(name)
        print("="*80)
        print("accuracy:", df["correct"].mean())
        print("final_answer_rate:", df["has_final_answer"].mean() if "has_final_answer" in df else None)
        print("avg_wall_s:", df["wall_s"].mean())
        print("avg_model_calls:", df["n_model_calls"].mean())
        print("avg_peak_gpu_mem_after_mb:", df["peak_gpu_mem_after_mb"].dropna().mean())
        if "completion_tokens_reported" in df:
            print("avg_completion_tokens_reported:", df["completion_tokens_reported"].dropna().mean())
        if "chosen_workflow" in df and df["chosen_workflow"].notna().any():
            print("chosen_workflow counts:")
            print(df["chosen_workflow"].value_counts(dropna=False))
            print("chosen_workflow accuracy:")
            print(df.groupby("chosen_workflow")["correct"].mean())

def pairwise_compare(results, a, b):
    if a not in results or b not in results:
        print(f"Missing {a} or {b}")
        return None
    da = results[a][["idx", "correct", "pred", "gold"]].rename(columns={"correct": f"{a}_correct", "pred": f"{a}_pred"})
    db = results[b][["idx", "correct", "pred"]].rename(columns={"correct": f"{b}_correct", "pred": f"{b}_pred"})
    m = da.merge(db, on="idx")
    table = pd.crosstab(m[f"{a}_correct"], m[f"{b}_correct"], rownames=[a], colnames=[b])
    print(table)
    print(f"{a} wrong -> {b} correct:", int(((m[f"{a}_correct"] == False) & (m[f"{b}_correct"] == True)).sum()))
    print(f"{a} correct -> {b} wrong:", int(((m[f"{a}_correct"] == True) & (m[f"{b}_correct"] == False)).sum()))
    return m

print_mode_diagnostics(results)
_ = pairwise_compare(results, "small_direct_finalized", "solver_verifier")
_ = pairwise_compare(results, "solver_verifier", "adaptive_scaffold_v2")

In [ ]:
#@title Failure inspection

for mode_name, df in results.items():
    print("\n" + "="*80)
    print(mode_name)
    print("="*80)
    fails = df[df["correct"] == False].head(5)
    if fails.empty:
        print("No failures in displayed sample.")
        continue
    for _, row in fails.iterrows():
        print(f"IDX={row['idx']} pred={row['pred']} gold={row['gold']} error={row['error']}")
        print(row["question"][:500].replace("\n", " "))
        print("chosen_workflow:", row.get("chosen_workflow"), "override:", row.get("override_reason"))
        print("-"*80)

## 11. Zip results

In [ ]:
#@title Zip results

zip_path = "/content/minnow_gguf_results_v2.zip"
import subprocess
subprocess.run(["zip", "-r", "-q", zip_path, str(RESULTS_DIR)], check=False)
print(zip_path)